# Autoencoders and image compression — dual framework

**DeepVision Masterclass — Modern Edition**

This notebook is designed as an executable teaching lab. Read the dataset contract and validation notes before training.

## 1. MNIST data and compression objective
Revisit the original linear and convolutional autoencoders. We compare reconstruction quality and bottleneck size, not classification accuracy.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch, tensorflow as tf
print(torch.__version__,tf.__version__)
(xtr,_),(xte,_)=tf.keras.datasets.mnist.load_data()
xtr=xtr.astype('float32')/255.; xte=xte.astype('float32')/255.
xtr=xtr[...,None]; xte=xte[...,None]
print(xtr.shape,xte.shape)

## 2. TensorFlow dense autoencoder
The bottleneck is a 32-dimensional latent vector. Binary cross-entropy is suitable for normalized grayscale pixels in [0,1].

In [ ]:
from tensorflow.keras import layers,Model
inp=layers.Input((28,28,1)); z=layers.Flatten()(inp)
z=layers.Dense(128,activation='relu')(z); z=layers.Dense(64,activation='relu')(z); latent=layers.Dense(32,activation='relu',name='latent')(z)
z=layers.Dense(64,activation='relu')(latent); z=layers.Dense(128,activation='relu')(z); z=layers.Dense(784,activation='sigmoid')(z); out=layers.Reshape((28,28,1))(z)
ae=Model(inp,out); encoder=Model(inp,latent)
ae.compile(optimizer='adam',loss='binary_crossentropy')
ae.summary()
# ae.fit(xtr,xtr,validation_split=.1,epochs=10,batch_size=256)

## 3. Convolutional autoencoder
Convolutions preserve spatial structure. Explicitly inspect output dimensions after each downsampling/upsampling operation.

In [ ]:
cin=layers.Input((28,28,1)); h=layers.Conv2D(32,3,padding='same',activation='relu')(cin); h=layers.MaxPooling2D(2,padding='same')(h); h=layers.Conv2D(16,3,padding='same',activation='relu')(h); code=layers.MaxPooling2D(2,padding='same',name='code')(h)
h=layers.Conv2D(16,3,padding='same',activation='relu')(code); h=layers.UpSampling2D(2)(h); h=layers.Conv2D(32,3,padding='same',activation='relu')(h); h=layers.UpSampling2D(2)(h); cout=layers.Conv2D(1,3,padding='same',activation='sigmoid')(h)
conv_ae=Model(cin,cout); conv_ae.compile(optimizer='adam',loss='binary_crossentropy'); conv_ae.summary()

## 4. PyTorch equivalent
NCHW layout is used by PyTorch. Note the output size can differ after strided operations; verify shapes before training.

In [ ]:
from torch import nn
class ConvAutoencoder(nn.Module):
 def __init__(self):
  super().__init__()
  self.enc=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
  self.dec=nn.Sequential(nn.Conv2d(16,16,3,padding=1),nn.ReLU(),nn.Upsample(scale_factor=2,mode='nearest'),nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.Upsample(scale_factor=2,mode='nearest'),nn.Conv2d(32,1,3,padding=1),nn.Sigmoid())
 def forward(self,x): return self.dec(self.enc(x))
pt_ae=ConvAutoencoder(); print(pt_ae(torch.randn(4,1,28,28)).shape)

## 5. Evaluate reconstruction
Report MSE/MAE and inspect examples. For perceptual or medical-image compression, add SSIM/LPIPS and task-specific downstream validation.

In [ ]:
from skimage.metrics import structural_similarity as ssim
# After training: recon=ae.predict(xte[:16]); compute per-image SSIM and visualize originals/reconstructions.
# Do not compare raw MSE across different intensity scaling or image resolutions.

## 6. Exercises
Change latent dimensions (8/16/32/64); compare dense vs convolutional encoders; add Gaussian noise for denoising; quantify compression ratio including metadata; test on Fashion-MNIST.